# 05 - Extraction Embeddings LLM (DistilBERT)

**SAE-112** — Epic 3 : Text Representation

## Objectif
Extraire des embeddings de documents à partir de **DistilBERT** pré-entraîné (HuggingFace).
Ces embeddings seront utilisés par le notebook ML classique (SAE-117).

**Grille :** LLM (1pt)

### Stratégie
- **Mean Pooling** : moyenne des hidden states de la dernière couche (hors padding)
- Dimension de sortie : **768** par document
- Dataset : **5 000 reviews** échantillonnées (faisable sur CPU)

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from tqdm import tqdm

device = torch.device('cpu')
print('Device :', device)
print('PyTorch :', torch.__version__)
print('Imports OK')

## 1. Chargement des données

In [ ]:
# Utilise reviews_text_cleaned si dispo, sinon reviews_clean
DATA_DIR = Path('../../data/cleaned')

path_cleaned = DATA_DIR / 'reviews_text_cleaned.parquet'
path_fallback = DATA_DIR / 'reviews_clean.parquet'

if path_cleaned.exists():
    reviews = pd.read_parquet(path_cleaned)
    text_col = 'text_cleaned' if 'text_cleaned' in reviews.columns else 'text'
    print(f'Chargé reviews_text_cleaned.parquet : {len(reviews):,} reviews')
elif path_fallback.exists():
    reviews = pd.read_parquet(path_fallback)
    text_col = 'text'
    print(f'Chargé reviews_clean.parquet : {len(reviews):,} reviews')
else:
    raise FileNotFoundError('Aucun fichier reviews trouvé dans data/cleaned/')

print(f'Colonne texte : {text_col}')
print(f'Colonnes : {reviews.columns.tolist()}')
reviews.head(3)

In [ ]:
# Echantillonnage stratifié sur les étoiles (5000 reviews)
N_SAMPLES = 5_000
RANDOM_STATE = 42

if len(reviews) > N_SAMPLES:
    # Essayer un echantillonnage stratifié sur les étoiles
    star_col = 'stars' if 'stars' in reviews.columns else 'stars_review' if 'stars_review' in reviews.columns else None
    if star_col:
        reviews = reviews.groupby(star_col, group_keys=False).apply(
            lambda x: x.sample(min(len(x), N_SAMPLES // 5), random_state=RANDOM_STATE)
        ).reset_index(drop=True)
        # Si pas assez, compléter jusqu'à N_SAMPLES
        if len(reviews) < N_SAMPLES:
            reviews = reviews.sample(min(len(reviews), N_SAMPLES), random_state=RANDOM_STATE).reset_index(drop=True)
    else:
        reviews = reviews.sample(N_SAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f'Echantillon : {len(reviews):,} reviews')

# Labels de polarité (0=négatif, 1=neutre, 2=positif)
star_col = 'stars' if 'stars' in reviews.columns else 'stars_review' if 'stars_review' in reviews.columns else None
if star_col:
    reviews['polarity'] = reviews[star_col].apply(
        lambda x: 0 if x <= 2 else (1 if x == 3 else 2)
    )
    print('Distribution de polarite :')
    print(reviews['polarity'].value_counts().sort_index().rename({0: 'Negatif', 1: 'Neutre', 2: 'Positif'}))
else:
    reviews['polarity'] = -1
    print('Colonne stars non trouvee, labels = -1')

## 2. Chargement de DistilBERT

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'

print(f'Chargement tokenizer {MODEL_NAME}...')
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

print(f'Chargement modele {MODEL_NAME}...')
model = DistilBertModel.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

print(f'Modele charge !')
print(f'  Parametres : {sum(p.numel() for p in model.parameters()):,}')
print(f'  Hidden size : {model.config.hidden_size}')

## 3. Extraction des embeddings

On utilise le **mean pooling** sur les hidden states de la dernière couche,
en masquant les tokens de padding.

In [ ]:
def mean_pooling(model_output, attention_mask):
    """Mean pooling sur les hidden states, masque du padding."""
    token_embeddings = model_output.last_hidden_state  # (batch, seq_len, 768)
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask


def extract_embeddings(texts, tokenizer, model, batch_size=32, max_length=128):
    """Extrait les embeddings DistilBERT (mean pooling) pour une liste de textes."""
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc='Extraction'):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_length, return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = model(**encoded)

        embeddings = mean_pooling(outputs, encoded['attention_mask'])
        all_embeddings.append(embeddings.cpu().numpy())

    return np.concatenate(all_embeddings, axis=0)


print('Fonctions d\'extraction definies')

In [ ]:
# max_length=128 : troncature agressive mais beaucoup plus rapide sur CPU
texts = reviews[text_col].fillna('').tolist()

print(f'Extraction des embeddings pour {len(texts):,} documents...')
print(f'Batch size=32 | Max length=128 (tronque pour vitesse sur CPU)')

embeddings = extract_embeddings(texts, tokenizer, model, batch_size=32, max_length=128)

print(f'\nEmbeddings extraits !')
print(f'  Shape  : {embeddings.shape}')
print(f'  Dtype  : {embeddings.dtype}')
print(f'  Memoire: {embeddings.nbytes / 1024 / 1024:.1f} MB')

## 4. Vérification rapide

In [ ]:
print('Statistiques des embeddings :')
print(f'  Moyenne : {embeddings.mean():.6f}')
print(f'  Std     : {embeddings.std():.6f}')
print(f'  Min     : {embeddings.min():.6f}')
print(f'  Max     : {embeddings.max():.6f}')

zero_vectors = np.sum(np.all(embeddings == 0, axis=1))
print(f'  Vecteurs nuls : {zero_vectors} / {len(embeddings)}')

print(f'\nPremier embedding (10 dims) : {embeddings[0][:10]}')

## 5. Visualisation t-SNE

In [ ]:
# PCA d'abord pour accélérer t-SNE
print('Reduction PCA (768 -> 50)...')
pca = PCA(n_components=50, random_state=42)
embeddings_pca = pca.fit_transform(embeddings)
print(f'Variance expliquee : {pca.explained_variance_ratio_.sum():.2%}')

# Sous-ensemble pour t-SNE (2000 points max pour la vitesse)
N_TSNE = min(2000, len(embeddings))
idx = np.random.default_rng(42).choice(len(embeddings), N_TSNE, replace=False)

print(f'Calcul t-SNE sur {N_TSNE} points...')
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=500)
embeddings_2d = tsne.fit_transform(embeddings_pca[idx])
print(f't-SNE termine : shape = {embeddings_2d.shape}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

polarities = reviews['polarity'].values[idx]
scatter = ax.scatter(
    embeddings_2d[:, 0], embeddings_2d[:, 1],
    c=polarities, cmap='RdYlGn', alpha=0.5, s=10
)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_ticks([0, 1, 2])
cbar.set_ticklabels(['Negatif', 'Neutre', 'Positif'])

ax.set_title('t-SNE des embeddings DistilBERT (5 000 reviews)', fontsize=14, fontweight='bold')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')

os.makedirs('../../outputs/figures', exist_ok=True)
plt.tight_layout()
plt.savefig('../../outputs/figures/tsne_distilbert.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardee : outputs/figures/tsne_distilbert.png')

## 6. Sauvegarde des embeddings

In [ ]:
output_dir = Path('../../data')
output_dir.mkdir(exist_ok=True)

emb_path = output_dir / 'distilbert_embeddings.npy'
lbl_path = output_dir / 'distilbert_labels.npy'

np.save(emb_path, embeddings)
print(f'Embeddings sauvegardes : {emb_path}')
print(f'  Shape : {embeddings.shape}')

labels = reviews['polarity'].values
np.save(lbl_path, labels)
print(f'Labels sauvegardes : {lbl_path}')
print(f'  Shape : {labels.shape}')
print(f'  Distribution : {dict(zip(*np.unique(labels, return_counts=True)))}')

print('\nNotebook SAE-112 termine avec succes !')
print(f'  {len(embeddings):,} embeddings DistilBERT de dimension {embeddings.shape[1]}')
print('  Fichiers prets pour SAE-117 (ML sur embeddings LLM)')